Mediapipe Setup 

In [1]:
import cv2
import numpy as np
import os
from matplotlib import pyplot as plt
import time
import mediapipe as mp

mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
mp_holistic = mp.solutions.holistic
mp_face_mesh = mp.solutions.face_mesh

def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # COLOR CONVERSION BGR 2 RGB
    image.flags.writeable = False                  # Image is no longer writeable
    results = model.process(image)                 # Make prediction
    image.flags.writeable = True                   # Image is now writeable
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR) # COLOR CONVERSION RGB 2 BGR
    return image, results

def draw_styled_landmarks(image, results):
    # Face landmarks
    if results.face_landmarks:
        mp_drawing.draw_landmarks(
            image,
            results.face_landmarks,
            mp_face_mesh.FACEMESH_TESSELATION,
            mp_drawing.DrawingSpec(color=(80,110,10), thickness=1, circle_radius=1),
            mp_drawing.DrawingSpec(color=(80,256,121), thickness=1, circle_radius=1)
        )
    if results.pose_landmarks:
        mp_drawing.draw_landmarks(
            image,
            results.pose_landmarks,
            mp_holistic.POSE_CONNECTIONS,
            mp_drawing.DrawingSpec(color=(80,22,10), thickness=2, circle_radius=4),
            mp_drawing.DrawingSpec(color=(80,44,121), thickness=2, circle_radius=2)
        )
    if results.left_hand_landmarks:
        mp_drawing.draw_landmarks(
            image,
            results.left_hand_landmarks,
            mp_holistic.HAND_CONNECTIONS,
            mp_drawing.DrawingSpec(color=(121,22,76), thickness=2, circle_radius=4),
            mp_drawing.DrawingSpec(color=(121,44,250), thickness=2, circle_radius=2)
        )
    if results.right_hand_landmarks:
        mp_drawing.draw_landmarks(
            image,
            results.right_hand_landmarks,
            mp_holistic.HAND_CONNECTIONS,
            mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4),
            mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2)
        )

def extract_keypoints(results):
    pose = np.array([[res.x, res.y, res.z, res.visibility] for res in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(33*4)
    face = np.array([[res.x, res.y, res.z] for res in results.face_landmarks.landmark]).flatten() if results.face_landmarks else np.zeros(468*3)
    lh = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*3)
    rh = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*3)
    return np.concatenate([pose, face, lh, rh])

Data Folder Creation and vocabulary Definition

In [2]:
import os
import numpy as np

# Path for exported data, numpy arrays
DATA_PATH = os.path.join('Dataset\MP_Data')

# Actions that we try to detect
actions = np.array(sorted(['hello']))

# Thirty videos worth of data
no_sequences = 30

# videos are going to be 30 frames in length
sequence_length = 30
start_folder = 1

for action in actions:
    action_path = os.path.join(DATA_PATH, action)
    
    # 1. Check if the action directory exists
    if not os.path.exists(action_path):
        # If not, create it and set dirmax to 0
        os.makedirs(action_path)
        dirmax = 0
    else:
        # 2. If it does exist, check if it is empty
        files = os.listdir(action_path)
        if len(files) == 0:
            dirmax = 0
        else:
            # 3. Calculate dirmax safely (ignoring system files like
            # .DS_Store or desktop.ini)
            valid_indices = []
            for f in files:
                try:
                    valid_indices.append(int(f))
                except ValueError:
                    pass # skip files that aren't numbers
            
            dirmax = np.max(valid_indices) if valid_indices else 0
            
    # Create the new sequence folders
    for sequence in range(1, no_sequences+1):
        try:
            os.makedirs(os.path.join(action_path, str(dirmax+sequence)))
        except:
            pass

Data Recording Code

In [4]:
import cv2
import numpy as np
import os
import time
import mediapipe as mp

# MediaPipe Setup
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils

def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image.flags.writeable = False
    results = model.process(image)
    image.flags.writeable = True
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    return image, results

def draw_styled_landmarks(image, results):
    if results.face_landmarks:
        mp_drawing.draw_landmarks(image, results.face_landmarks, mp_holistic.FACEMESH_TESSELATION,
                                 mp_drawing.DrawingSpec(color=(80,110,10), thickness=1, circle_radius=1),
                                 mp_drawing.DrawingSpec(color=(80,256,121), thickness=1, circle_radius=1))
    if results.pose_landmarks:
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS,
                                 mp_drawing.DrawingSpec(color=(80,22,10), thickness=2, circle_radius=4),
                                 mp_drawing.DrawingSpec(color=(80,44,121), thickness=2, circle_radius=2))
    if results.left_hand_landmarks:
        mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS,
                                 mp_drawing.DrawingSpec(color=(121,22,76), thickness=2, circle_radius=4),
                                 mp_drawing.DrawingSpec(color=(121,44,250), thickness=2, circle_radius=2))
    if results.right_hand_landmarks:
        mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS,
                                 mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4),
                                 mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2))

def extract_keypoints(results):
    pose = np.array([[res.x, res.y, res.z, res.visibility] for res in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(33*4)
    face = np.array([[res.x, res.y, res.z] for res in results.face_landmarks.landmark]).flatten() if results.face_landmarks else np.zeros(468*3)
    lh = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*3)
    rh = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*3)
    return np.concatenate([pose, face, lh, rh])

# Configuration
DATA_PATH = os.path.join('Dataset', 'MP_Data')
actions = np.array(['hello']) # Change or add words here e.g. ['hello', 'thankyou']
no_sequences_to_add = 30      # How many NEW videos to record in this run
sequence_length = 30          # 30 frames per video (~1 second)

cap = cv2.VideoCapture(0)

with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    for action in actions:
        action_path = os.path.join(DATA_PATH, action)
        
        # Determine starting folder index automatically
        if os.path.exists(action_path):
            existing_folders = [int(f) for f in os.listdir(action_path) if f.isdigit()]
            start_folder = max(existing_folders) + 1 if existing_folders else 0
        else:
            os.makedirs(action_path)
            start_folder = 0
            
        print(f"Recording '{action}' starting at Sequence #{start_folder}")

        for sequence in range(start_folder, start_folder + no_sequences_to_add):
            
            # --- PHASE 1: GET READY COUNTDOWN (2 Seconds) ---
            countdown_seconds = 2
            start_time = time.time()
            while time.time() - start_time < countdown_seconds:
                ret, frame = cap.read()
                if not ret: break
                
                image, results = mediapipe_detection(frame, holistic)
                draw_styled_landmarks(image, results)
                
                time_left = int(countdown_seconds - (time.time() - start_time)) + 1
                
                # Header background bar
                cv2.rectangle(image, (0, 0), (640, 50), (50, 50, 50), -1)
                
                # Large, readable text
                cv2.putText(image, f'Action: {action.upper()} | Sequence #{sequence}', (10, 35), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2, cv2.LINE_AA)
                
                # Countdown prompt on screen
                cv2.putText(image, f'GET READY: {time_left}', (160, 250), 
                            cv2.FONT_HERSHEY_SIMPLEX, 1.8, (0, 255, 255), 4, cv2.LINE_AA)
                
                cv2.imshow('OpenCV Feed', image)
                cv2.waitKey(1)

            # --- PHASE 2: ACTIVE RECORDING (30 Frames) ---
            for frame_num in range(sequence_length):
                ret, frame = cap.read()
                if not ret: break
                
                image, results = mediapipe_detection(frame, holistic)
                draw_styled_landmarks(image, results)
                
                # Extract and save keypoints
                keypoints = extract_keypoints(results)
                folder_path = os.path.join(DATA_PATH, action, str(sequence))
                if not os.path.exists(folder_path):
                    os.makedirs(folder_path)
                np.save(os.path.join(folder_path, str(frame_num)), keypoints)

                # Header background bar (Red during recording)
                cv2.rectangle(image, (0, 0), (640, 50), (0, 0, 180), -1)
                
                # Action info
                cv2.putText(image, f'Action: {action.upper()} | Sequence #{sequence}', (10, 35), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2, cv2.LINE_AA)
                
                # Active Frame Counter overlay
                cv2.putText(image, f'RECORDING: Frame {frame_num + 1}/{sequence_length}', (100, 250), 
                            cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3, cv2.LINE_AA)
                
                cv2.imshow('OpenCV Feed', image)
                
                if cv2.waitKey(10) & 0xFF == ord('q'):
                    break

cap.release()
cv2.destroyAllWindows()

Recording 'hello' starting at Sequence #0


Preprocessing

In [5]:
# Preprocessing
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

# Create a dictionary mapping each string label to a number
label_map = {label: num for num, label in enumerate(actions)}

sequences, labels = [], []
for action in actions:
    folder_path = os.path.join(DATA_PATH, action)
    
    # Safely get all sequence folders and sort them numerically
    sequence_folders = sorted(
        [int(x) for x in os.listdir(folder_path)],
        key=lambda x: x
    )
    
    for sequence in sequence_folders:
        window = [] # This will hold the 30 frames for a single video
        for frame_num in range(sequence_length):
            # Load each individual frame's numpy array from disk
            res = np.load(os.path.join(folder_path, str(sequence), f"{frame_num}.npy"))
            window.append(res)
            
        # Append the full 30-frame window to our master list
        sequences.append(window)
        # Append the corresponding numerical label
        labels.append(label_map[action])

Training Code

In [6]:
# Training Code
import numpy as np
import os
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Bidirectional, LSTM, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

# 1. Force Reload Data (To ensure no Augmentation)
sequences_ph = []
labels_ph = []

for action in actions:
    folder_path = os.path.join(DATA_PATH, action)
    sequence_folders = sorted([int(x) for x in os.listdir(folder_path)], key=lambda x: x)
    
    for sequence in sequence_folders:
        window = []
        for frame_num in range(sequence_length):
            res = np.load(os.path.join(folder_path, str(sequence), f"{frame_num}.npy"))
            
            # Filter: Pose (0-132) + Hands (1536-end)
            pose = res[:132]
            # Hands are the last part of the full feature vector
            # The full vector is 132 (pose) + 1404 (face) + 63 (lh) + 63 (rh) = 1662
            # Face starts at 132, Hands start at 1536 (132+1404=1536)
            hands = res[1536:]
            
            window.append(np.concatenate([pose, hands]))
            
        sequences_ph.append(window)
        labels_ph.append(label_map[action])

# 2. Process Data
X_clean = np.array(sequences_ph)
y_clean = to_categorical(labels_ph).astype(int)

# Normalize
mu = np.mean(X_clean, axis=0)
sigma = np.std(X_clean, axis=0) + 1e-6
X_clean = (X_clean - mu) / sigma

# Split (Standard 95/5 split)
X_train_clean, X_test_clean, y_train_clean, y_test_clean = train_test_split(
    X_clean, y_clean, test_size=0.05
)
print(f"Training on CLEAN data shape: {X_train_clean.shape}")

# 3. Define The Model
model_ph = Sequential([
    # input_shape is (30, 258) where 258 = 132 (pose) + 126 (hands)
    Bidirectional(LSTM(128, return_sequences=True, input_shape=(30, 258))), 
    Bidirectional(LSTM(64)),
    Dense(64, activation='relu'),
    Dense(len(actions), activation='softmax') # Corrected output size to len(actions) = 7
])

# 4. Compile & Train
# We lower learning rate slightly to 0.0005 to prevent crashing
optimizer_ph = Adam(learning_rate=0.0005)
model_ph.compile(optimizer=optimizer_ph, loss='categorical_crossentropy', metrics=['accuracy'])

# Train until we see

Training on CLEAN data shape: (28, 30, 258)



Validation and Evaluation Code

In [7]:
# Validation and Evaluation Code
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Get predictions
y_pred = model_ph.predict(X_test_clean)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_test_clean, axis=1)

# 1. Print F1, Precision, Recall
print(classification_report(y_true, y_pred_classes, target_names=actions))

# 2. Generate Confusion Matrix Plot
cm = confusion_matrix(y_true, y_pred_classes)
plt.figure(figsize=(10,7))
sns.heatmap(
    cm, 
    annot=True, 
    xticklabels=actions, 
    yticklabels=actions,
    fmt='d'
)
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix - 7 Class SLD')
plt.show()

1/1 [==============================] - 5s 5s/step
              precision    recall  f1-score   support

       hello       1.00      1.00      1.00         2

    accuracy                           1.00         2
   macro avg       1.00      1.00      1.00         2
weighted avg       1.00      1.00      1.00         2



c:\01 - Uday\02. IITJ\MTech Project\RealTimeSignLanguageTranslator\.venv\Lib\site-packages\sklearn\metrics\_classification.py:386: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


<Figure size 1000x700 with 2 Axes>

Real Time Testing Code


In [ ]:
# Real Time Testing Code
# with Hands limitation (Does not allow the model to predict unless hand 
# keypoints are present in the frame and loads the model from a saved file)
import cv2
import numpy as np
import mediapipe as mp
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tensorflow.keras.models import load_model 
import time
import os

print("⏳ Loading Models...")

# --- A. Load Grammar AI ---
try:
    tokenizer = AutoTokenizer.from_pretrained("vennify/t5-base-grammar-correction")
    grammar_model = AutoModelForSeq2SeqLM.from_pretrained("vennify/t5-base-grammar-correction")
    print("Grammar AI Loaded!")
except Exception as e:
    print(f"Error loading Grammar AI: {e}")

# --- B. Load Keras Sign Language Model ---
MODEL_PATH = "model_ph_clean_100.keras" 
try:
    if os.path.exists(MODEL_PATH):
        model_ph = load_model(MODEL_PATH)
        print(f"Sign Language Model Loaded: {MODEL_PATH}")
    else:
        raise FileNotFoundError(f"Model file not found: {MODEL_PATH}")
except Exception as e:
    print(f"Error loading Keras Model: {e}")

# --- C. Load Normalization Data (Mu & Sigma) ---
try:
    if os.path.exists('mu.npy') and os.path.exists('sigma.npy'):
        mu = np.load('mu.npy')
        sigma = np.load('sigma.npy')
        print("Normalization parameters (mu/sigma) loaded!")
    else:
        print("WARNING: 'mu.npy' or 'sigma.npy' not found.")
except Exception as e:
    print(f"❌ Error loading normalization data: {e}")

def ai_correct_sentence(raw_words):
    if not raw_words: return ""
    raw_text = " ".join(raw_words)
    input_text = f"grammar: {raw_text}"
    inputs = tokenizer(input_text, return_tensors="pt")
    result = grammar_model.generate(**inputs, max_length=128)
    return tokenizer.decode(result[0], skip_special_tokens=True)

# Ensure these match exactly what you trained on
actions = np.array(sorted(['hello', 'thankyou', 'are', 'what', 'name', 'beautiful','your']))
colors = [(245,117,16), (117,245,16), (16,117,245), (200, 50, 200), (50, 200, 200), (200, 200, 50), (200, 50, 50)]
threshold = 0.85
SAVE_FILE = "notes.txt"

if 'mu' not in globals() or 'sigma' not in globals():
    raise ValueError("ERROR: 'mu' and 'sigma' are missing.")

# MediaPipe Setup
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils

def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image.flags.writeable = False
    results = model.process(image)
    image.flags.writeable = True
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    return image, results

def draw_styled_landmarks(image, results):
    if results.face_landmarks:
        mp_drawing.draw_landmarks(image, results.face_landmarks, mp_holistic.FACEMESH_TESSELATION,
                                 mp_drawing.DrawingSpec(color=(80,110,10), thickness=1, circle_radius=1),
                                 mp_drawing.DrawingSpec(color=(80,256,121), thickness=1, circle_radius=1))
    if results.pose_landmarks:
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS,
                                 mp_drawing.DrawingSpec(color=(80,22,10), thickness=2, circle_radius=4),
                                 mp_drawing.DrawingSpec(color=(80,44,121), thickness=2, circle_radius=2))
    if results.left_hand_landmarks:
        mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS,
                                 mp_drawing.DrawingSpec(color=(121,22,76), thickness=2, circle_radius=4),
                                 mp_drawing.DrawingSpec(color=(121,44,250), thickness=2, circle_radius=2))
    if results.right_hand_landmarks:
        mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS,
                                 mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4),
                                 mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2))

def prob_viz(res, actions, input_frame, colors):
    output_frame = input_frame.copy()
    for num, prob in enumerate(res):
        if num >= len(colors): break
        cv2.rectangle(output_frame, (0, 60 + num*40), (int(prob*200), 90 + num*40), colors[num], -1)
        cv2.putText(output_frame, actions[num], (0, 85 + num*40), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2, cv2.LINE_AA)
    return output_frame

def extract_keypoints_ph(results):
    pose = np.array([[res.x, res.y, res.z, res.visibility] for res in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(33*4)
    lh = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*3)
    rh = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*3)
    return np.concatenate([pose, lh, rh])

sequence_ph = []
sentence_ph = []
predictions_ph = []
corrected_sentence = ""

save_notification_timer = 0
show_save_msg = False

cap = cv2.VideoCapture(0)
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        
        # Detection
        image, results = mediapipe_detection(frame, holistic)
        draw_styled_landmarks(image, results)
        
        # Hands Limitation Check
        hands_present = results.left_hand_landmarks or results.right_hand_landmarks
        
        if hands_present:
            keypoints = extract_keypoints_ph(results)
            sequence_ph.append(keypoints)
            sequence_ph = sequence_ph[-30:]
            
            if len(sequence_ph) == 30:
                input_data = (np.array(sequence_ph) - mu) / sigma
                res = model_ph.predict(np.expand_dims(input_data, axis=0))[0]
                predictions_ph.append(np.argmax(res))
                
                # Stability Check
                if len(predictions_ph) > 10:
                    if np.unique(predictions_ph[-10:])[0] == np.argmax(res):
                        if res[np.argmax(res)] > threshold:
                            predicted_action = actions[np.argmax(res)]
                            
                            if len(sentence_ph) == 0 or predicted_action != sentence_ph[-1]:
                                sentence_ph.append(predicted_action)
                                corrected_sentence = ai_correct_sentence(sentence_ph[-5:])
                                
                                # Keep only the last 5 words to prevent unbounded growth
                                sentence_ph = sentence_ph[-5:]
                                
            image = prob_viz(res, actions, image, colors)
        else:
            cv2.putText(image, "NO HANDS DETECTED", (15, 200), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2, cv2.LINE_AA)

        # UI Overlays
        cv2.rectangle(image, (0,0), (640, 40), (245, 117, 16), -1)
        cv2.putText(image, "Raw: " + ' '.join(sentence_ph), (3,30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)
        
        cv2.rectangle(image, (0, 440), (640, 480), (35, 35, 35), -1)
        cv2.putText(image, corrected_sentence, (3, 470), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2, cv2.LINE_AA)
        
        # Save Notification Logic
        if show_save_msg:
            cv2.putText(image, "SAVED TO FILE!", (200, 240), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 0), 3, cv2.LINE_AA)
            if time.time() - save_notification_timer > 1:
                show_save_msg = False
                
        cv2.imshow('OpenCV Feed (Enter to Save)', image)
        
        key = cv2.waitKey(10) & 0xFF
        if key == ord('q'): 
            break
        elif key == 13: # Enter Key
            if corrected_sentence:
                with open(SAVE_FILE, "a") as f:
                    f.write(corrected_sentence + "\n")
                print(f"Saved: {corrected_sentence}")
                show_save_msg = True
                save_notification_timer = time.time()
                
    cap.release()
    cv2.destroyAllWindows()

c:\01 - Uday\02. IITJ\MTech Project\RealTimeSignLanguageTranslator\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]


⏳ Loading Models...


c:\01 - Uday\02. IITJ\MTech Project\RealTimeSignLanguageTranslator\.venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
c:\01 - Uday\02. IITJ\MTech Project\RealTimeSignLanguageTranslator\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jakat\.cache\huggingface\hub\models--vennify--t5-base-grammar-correction. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support sy

Grammar AI Loaded!
Error loading Keras Model: Model file not found: model_ph_clean_100.keras


c:\01 - Uday\02. IITJ\MTech Project\RealTimeSignLanguageTranslator\.venv\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


1/1 [==============================] - 0s 41ms/step


KeyboardInterrupt: 

: 